# D3 - OOP for Pipelines: Composition Over Inheritance

## Objective
Demonstrate object-oriented design principles applied to data pipelines. Explore abstract base classes, concrete processing steps, composition over inheritance, and dynamic runtime step replacement without modifying core pipeline code.

## Concepts Covered
- **Abstract Base Classes (`ABC`, `@abstractmethod`)**: Standardizing step contracts using `Step`.
- **Composition Over Inheritance**: Assembling flexible processing pipelines by composing independent step instances.
- **Encapsulation & Polymorphism**: Hiding stage-specific logic while treating all processing stages uniformly.
- **Runtime Extensibility & Swapping**: Dynamically altering pipeline workflows at runtime without altering core classes (Open/Closed Principle).

## Project Implementation
The pipeline components reside in `app/services/pipeline.py` and `app/services/pipeline_stages.py`:
- `Step` (ABC): Defines the abstract `process(data: dict) -> dict` method.
- `TaskValidationStep`: Validates task fields (title, priority, status).
- `TaskTransformationStep`: Normalizes text, sets timestamps, cleans fields.
- `TaskProcessingStep`: Computes status metadata and execution metrics.
- `Pipeline`: Manages and executes an ordered list of `Step` implementations.

## Demonstration
We demonstrate building a pipeline, processing payload data, adding custom steps, and swapping steps dynamically at runtime.

In [1]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.services.pipeline import Pipeline
from app.services.pipeline_stages import (
    Step,
    TaskValidationStep,
    TaskTransformationStep,
    TaskProcessingStep,
)

# Create standard default pipeline
pipeline = Pipeline([
    TaskValidationStep(),
    TaskTransformationStep(),
    TaskProcessingStep()
])

print(f"Pipeline initial step count: {len(pipeline.steps)}")
for s in pipeline.steps:
    print(f" - Step: {s.__class__.__name__}")

# Execute pipeline on sample data
sample_task = {
    "title": "  Implement D3 Composition  ",
    "priority": "HIGH",
    "status": "pending"
}

result = pipeline.run(sample_task)
print("\nProcessed Pipeline Result:")
for k, v in result.items():
    print(f"  {k}: {v}")

Pipeline initial step count: 3
 - Step: TaskValidationStep
 - Step: TaskTransformationStep
 - Step: TaskProcessingStep

Processed Pipeline Result:
  title: Implement D3 Composition
  priority: HIGH
  status: PROCESSED
  completed: False
  processed: True


In [2]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.services.pipeline import Pipeline
from app.services.pipeline_stages import Step

# Demonstrate Adding a New Step Without Modifying Pipeline Source Code
class CustomAuditStep(Step):
    def process(self, data: dict) -> dict:
        data["audit_flag"] = True
        data["audited_by"] = "D3_Demonstration"
        return data

pipeline.add_step(CustomAuditStep())

print(f"Pipeline updated step count: {len(pipeline.steps)}")
result_audited = pipeline.run(sample_task)
print(f"Audit Flag Present: {result_audited.get('audit_flag')}")
print(f"Audited By: {result_audited.get('audited_by')}")

Pipeline updated step count: 4
Audit Flag Present: True
Audited By: D3_Demonstration


In [3]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.services.pipeline_stages import Step

# Demonstrate Runtime Step Swapping
class StrictValidationStep(Step):
    def process(self, data: dict) -> dict:
        if not data.get("title"):
            raise ValueError("Title is strictly required!")
        data["strict_validated"] = True
        return data

# Replace TaskValidationStep (index 0) with StrictValidationStep
print("\nSwapping validation step at runtime...")
pipeline._steps[0] = StrictValidationStep()

for i, s in enumerate(pipeline.steps):
    print(f" Step {i+1}: {s.__class__.__name__}")

swapped_result = pipeline.run(sample_task)
print(f"Strict Validated: {swapped_result.get('strict_validated')}")


Swapping validation step at runtime...
 Step 1: StrictValidationStep
 Step 2: TaskTransformationStep
 Step 3: TaskProcessingStep
 Step 4: CustomAuditStep
Strict Validated: True


## Actual Output
The code cells demonstrate:
1. Sequential execution of composed steps (`TaskValidationStep` $\rightarrow$ `TaskTransformationStep` $\rightarrow$ `TaskProcessingStep`).
2. Seamless extension by creating `CustomAuditStep` without modifying `Pipeline`.
3. Dynamic step replacement by swapping step instances at index 0 at runtime.

## Key Observations
- Composition allows flexible assembly of pipelines at runtime without deeply nested inheritance hierarchies.
- New processing requirements are implemented by creating new `Step` subclasses, fulfilling the Open/Closed Principle.
- Each stage remains encapsulated and loosely coupled.

## Conclusion
Using composition over inheritance for data pipelines ensures high modularity, testability, and runtime adaptability.